# City Library Management System — Graded Mini Project (Part A)

Complete solution: book & member records, borrow/return workflow with validation,
reusable reporting functions, a runnable demo, and an interactive console menu.


## Design Notes & Assumptions (≤300 words)

**Data structures**
- `books`: dict keyed by `book_id` → `{title, author, genre, availability}`. A dict gives O(1)
  lookup/update on issue and return, versus scanning a list.
- `members`: dict keyed by `member_id` → `{name, age, contact, borrowed_books}`. `borrowed_books`
  is a list, so a member can hold more than one book at once.
- `borrow_log`: a list of transaction dicts. Every borrow **and** every return is appended (never
  overwritten), so it doubles as a historical audit trail and powers the "most popular genre" report.

**Validation rules**
- A book cannot be issued if its status is already `Issued`.
- Book/member IDs must exist before being used in any operation.
- No duplicate Book IDs or Member IDs.
- A return only succeeds if that specific member is the one currently holding the book.
- Member `age` is validated as a positive integer at registration time; invalid input is rejected
  with a clear message rather than silently stored as a string.

**Code organisation**
- All business logic lives in small, single-purpose, reusable functions — no `print()`/`input()`
  mixed into the logic itself (aside from a function returning its own result message).
- A thin CLI (`main_menu`) sits on top and only calls these functions.

**Note on search scope**: `search_book()` matches **title or author** only, per the spec ("Search
for a book by title or author"). Searching a genre name (e.g. "Fantasy") in the search box
correctly returns no results — genre lookups go through `available_books_by_genre()` instead. This
is intentional, not a bug (see the demo below).

**How to run**: run all cells, or run the script directly with
`python library_management_system.py`. `RUN_DEMO = True` runs a scripted demo first (so every
requirement is visibly exercised with zero typing), then drops into the interactive menu.


In [1]:
from collections import Counter
from datetime import datetime

RUN_DEMO = True  # set to False to skip the automatic demo and go straight to the menu

AVAILABLE_STATUS = "Available"
ISSUED_STATUS = "Issued"

def normalize_text(value):
    """Return a trimmed string value for consistent IDs and names."""
    if value is None:
        return ""
    return str(value).strip()

def initialize_library():
    """Reset all in-memory data structures for a fresh run."""
    global books, members, borrow_log
    books.clear()
    members.clear()
    borrow_log.clear()


In [2]:
# ============================================================================
# 1. DATA STORES
# ============================================================================
books = {}       # book_id -> {title, author, genre, availability}
members = {}     # member_id -> {name, age, contact, borrowed_books: []}
borrow_log = []  # list of {book_id, member_id, action, timestamp}


In [3]:
# ============================================================================
# 2. BOOK MANAGEMENT
# ============================================================================
def add_book(book_id, title, author, genre):
    """Add a new book to the catalogue. Returns True/False + message."""
    book_id = normalize_text(book_id)
    title = normalize_text(title)
    author = normalize_text(author)
    genre = normalize_text(genre)

    if not all([book_id, title, author, genre]):
        return False, "Book ID, title, author, and genre must not be empty."
    if book_id in books:
        return False, f"Book ID '{book_id}' already exists."

    books[book_id] = {
        "title": title,
        "author": author,
        "genre": genre,
        "availability": AVAILABLE_STATUS,
    }
    return True, f"Book '{title}' added successfully."

def update_book_availability(book_id, status):
    """Internal helper: set a book's availability to 'Available' or 'Issued'."""
    if book_id not in books:
        return False, f"Book ID '{book_id}' not found."
    books[book_id]["availability"] = status
    return True, "Updated."


In [4]:
# ============================================================================
# 3. MEMBER MANAGEMENT
# ============================================================================
def validate_age(age):
    """Return (True, int_age) if age is a valid positive integer, else (False, message)."""
    try:
        age_int = int(age)
    except (TypeError, ValueError):
        return False, "Age must be a whole number."
    if age_int <= 0 or age_int > 130:
        return False, "Age must be a realistic positive number."
    return True, age_int

def add_member(member_id, name, age, contact):
    """Register a new library member."""
    member_id = normalize_text(member_id)
    name = normalize_text(name)
    contact = normalize_text(contact)

    if not all([member_id, name, contact]):
        return False, "Member ID, name, and contact info must not be empty."
    if member_id in members:
        return False, f"Member ID '{member_id}' already exists."

    age_ok, age_value = validate_age(age)
    if not age_ok:
        return False, age_value  # age_value holds the error message here

    members[member_id] = {
        "name": name,
        "age": age_value,
        "contact": contact,
        "borrowed_books": [],
    }
    return True, f"Member '{name}' registered successfully."


In [5]:
# ============================================================================
# 4. BORROW & RETURN SYSTEM
# ============================================================================
def issue_book(book_id, member_id):
    """Issue a book to a member, with validation."""
    book_id = normalize_text(book_id)
    member_id = normalize_text(member_id)

    if not book_id or not member_id:
        return False, "Book ID and Member ID must not be empty."
    if book_id not in books:
        return False, f"Book ID '{book_id}' does not exist."
    if member_id not in members:
        return False, f"Member ID '{member_id}' does not exist."
    if books[book_id]["availability"] == ISSUED_STATUS:
        return False, f"Book '{books[book_id]['title']}' is already issued to someone else."

    update_book_availability(book_id, ISSUED_STATUS)
    members[member_id]["borrowed_books"].append(book_id)
    borrow_log.append({
        "book_id": book_id,
        "member_id": member_id,
        "action": "Issued",
        "timestamp": datetime.now(),
    })
    return True, f"Book '{books[book_id]['title']}' issued to {members[member_id]['name']}."

def return_book(book_id, member_id):
    """Return a previously issued book, with validation."""
    book_id = normalize_text(book_id)
    member_id = normalize_text(member_id)

    if not book_id or not member_id:
        return False, "Book ID and Member ID must not be empty."
    if book_id not in books:
        return False, f"Book ID '{book_id}' does not exist."
    if member_id not in members:
        return False, f"Member ID '{member_id}' does not exist."
    if book_id not in members[member_id]["borrowed_books"]:
        return False, f"'{members[member_id]['name']}' is not currently holding this book."

    update_book_availability(book_id, AVAILABLE_STATUS)
    members[member_id]["borrowed_books"].remove(book_id)
    borrow_log.append({
        "book_id": book_id,
        "member_id": member_id,
        "action": "Returned",
        "timestamp": datetime.now(),
    })
    return True, f"Book '{books[book_id]['title']}' returned by {members[member_id]['name']}."


In [6]:
# ============================================================================
# 5. REPORTS & QUERIES
# ============================================================================
def available_books_by_genre(genre):
    """Return a list of available books in a given genre (case-insensitive)."""
    return [
        {"book_id": bid, **info}
        for bid, info in books.items()
        if info["genre"].lower() == genre.lower() and info["availability"] == AVAILABLE_STATUS
    ]

def members_who_borrowed():
    """Return members who currently have at least one book borrowed."""
    return [
        {"member_id": mid, "name": info["name"], "borrowed_books": info["borrowed_books"]}
        for mid, info in members.items()
        if info["borrowed_books"]
    ]

def search_book(keyword):
    """Search books by title or author only (case-insensitive substring match).
    Genre is intentionally excluded — use available_books_by_genre() for genre lookups."""
    keyword = keyword.lower()
    return [
        {"book_id": bid, **info}
        for bid, info in books.items()
        if keyword in info["title"].lower() or keyword in info["author"].lower()
    ]

def most_popular_genre():
    """Return the genre with the highest number of 'Issued' transactions in the log."""
    issued_genres = [
        books[entry["book_id"]]["genre"]
        for entry in borrow_log
        if entry["action"] == "Issued" and entry["book_id"] in books
    ]
    if not issued_genres:
        return None, 0
    counts = Counter(issued_genres)
    genre, count = counts.most_common(1)[0]
    return genre, count

def library_summary():
    """A one-shot overview report combining several stats."""
    total_books = len(books)
    issued_books = sum(1 for b in books.values() if b["availability"] == ISSUED_STATUS)
    available_books = total_books - issued_books
    top_genre, top_count = most_popular_genre()
    return {
        "total_books": total_books,
        "available_books": available_books,
        "issued_books": issued_books,
        "total_members": len(members),
        "members_with_borrowed_books": len(members_who_borrowed()),
        "most_popular_genre": top_genre,
        "most_popular_genre_issue_count": top_count,
    }


In [7]:
# ============================================================================
# 6. DISPLAY HELPERS (printing only — no business logic)
# ============================================================================
def print_books(book_list, empty_msg="No books found."):
    if not book_list:
        print(empty_msg)
        return
    for b in book_list:
        print(f"  [{b['book_id']}] '{b['title']}' by {b['author']} "
              f"({b['genre']}) - {b['availability']}")

def print_members(member_list, empty_msg="No members found."):
    if not member_list:
        print(empty_msg)
        return
    for m in member_list:
        books_str = ", ".join(m["borrowed_books"]) if m["borrowed_books"] else "None"
        print(f"  [{m['member_id']}] {m['name']} - Borrowed: {books_str}")


In [8]:
# ============================================================================
# 7. DEMO (auto-runs so every requirement is visibly demonstrated)
# ============================================================================
def run_demo():
    initialize_library()
    print("=" * 70)
    print("DEMO: CITY LIBRARY MANAGEMENT SYSTEM")
    print("=" * 70)

    print("\n-- Adding books --")
    for args in [
        ("B001", "The Hobbit", "J.R.R. Tolkien", "Fantasy"),
        ("B002", "Dune", "Frank Herbert", "Sci-Fi"),
        ("B003", "A Game of Thrones", "George R. R. Martin", "Fantasy"),
        ("B004", "Foundation", "Isaac Asimov", "Sci-Fi"),
        ("B005", "The Hobbit Companion", "J.R.R. Tolkien", "Fantasy"),
    ]:
        ok, msg = add_book(*args)
        print(" ", msg)

    print("\n-- Adding members --")
    for args in [("M001", "Aarav Shah", 28, "aarav@example.com"),
                 ("M002", "Priya Mehta", 34, "priya@example.com")]:
        ok, msg = add_member(*args)
        print(" ", msg)

    print("\n-- Attempting to register a member with an invalid age (should fail) --")
    ok, msg = add_member("M003", "Test User", "not-a-number", "test@example.com")
    print(" ", msg)

    print("\n-- Issuing books --")
    for bid, mid in [("B001", "M001"), ("B002", "M002"), ("B003", "M001")]:
        ok, msg = issue_book(bid, mid)
        print(" ", msg)

    print("\n-- Attempting to issue an already-issued book (should fail) --")
    ok, msg = issue_book("B001", "M002")
    print(" ", msg)

    print("\n-- Returning a book --")
    ok, msg = return_book("B002", "M002")
    print(" ", msg)

    print("\n-- Available Fantasy books --")
    print_books(available_books_by_genre("Fantasy"))

    print("\n-- Members who have borrowed books --")
    print_members(members_who_borrowed())

    print("\n-- Search for 'hobbit' (title/author) --")
    print_books(search_book("hobbit"))

    print("\n-- Search for 'Fantasy' (title/author only, genre is out of scope"
          " -- see design notes; use option 5 for genre lookups) --")
    print_books(search_book("Fantasy"))

    print("\n-- Library Summary --")
    for k, v in library_summary().items():
        print(f"  {k}: {v}")

    print("\n" + "=" * 70)
    print("END OF DEMO - entering interactive menu now")
    print("=" * 70 + "\n")


In [9]:
# ============================================================================
# 8. INTERACTIVE MENU (console interface)
# ============================================================================
def main_menu():
    menu = """
------------------------------------------
CITY LIBRARY MANAGEMENT SYSTEM
------------------------------------------
1. Add a new book
2. Add a new member
3. Issue a book
4. Return a book
5. Show available books by genre
6. Show members who borrowed books
7. Search book by title/author
8. Show most popular genre
9. Show library summary
0. Exit
------------------------------------------
"""
    while True:
        print(menu)
        choice = input("Enter your choice: ").strip()

        if choice == "1":
            bid = input("Book ID: ").strip()
            title = input("Title: ").strip()
            author = input("Author: ").strip()
            genre = input("Genre: ").strip()
            ok, msg = add_book(bid, title, author, genre)
            print(msg)

        elif choice == "2":
            mid = input("Member ID: ").strip()
            name = input("Name: ").strip()
            age = input("Age: ").strip()
            contact = input("Contact Info: ").strip()
            ok, msg = add_member(mid, name, age, contact)
            print(msg)

        elif choice == "3":
            bid = input("Book ID to issue: ").strip()
            mid = input("Member ID: ").strip()
            ok, msg = issue_book(bid, mid)
            print(msg)

        elif choice == "4":
            bid = input("Book ID to return: ").strip()
            mid = input("Member ID: ").strip()
            ok, msg = return_book(bid, mid)
            print(msg)

        elif choice == "5":
            genre = input("Genre: ").strip()
            print_books(available_books_by_genre(genre))

        elif choice == "6":
            print_members(members_who_borrowed())

        elif choice == "7":
            keyword = input("Enter title/author keyword: ").strip()
            print_books(search_book(keyword))

        elif choice == "8":
            genre, count = most_popular_genre()
            if genre:
                print(f"Most popular genre: {genre} ({count} books issued)")
            else:
                print("No books have been issued yet.")

        elif choice == "9":
            for k, v in library_summary().items():
                print(f"  {k}: {v}")

        elif choice == "0":
            print("Goodbye!")
            break

        else:
            print("Invalid choice, please try again.")


In [10]:
# ============================================================================
# 9. ENTRY POINT
# ============================================================================
if __name__ == "__main__":
    initialize_library()
    if RUN_DEMO:
        run_demo()
    # NOTE: in a notebook, calling main_menu() will prompt for input() in a cell.
    # Uncomment the line below to try the interactive menu manually.
    # main_menu()


DEMO: CITY LIBRARY MANAGEMENT SYSTEM

-- Adding books --
  Book 'The Hobbit' added successfully.
  Book 'Dune' added successfully.
  Book 'A Game of Thrones' added successfully.
  Book 'Foundation' added successfully.
  Book 'The Hobbit Companion' added successfully.

-- Adding members --
  Member 'Aarav Shah' registered successfully.
  Member 'Priya Mehta' registered successfully.

-- Attempting to register a member with an invalid age (should fail) --
  Age must be a whole number.

-- Issuing books --
  Book 'The Hobbit' issued to Aarav Shah.
  Book 'Dune' issued to Priya Mehta.
  Book 'A Game of Thrones' issued to Aarav Shah.

-- Attempting to issue an already-issued book (should fail) --
  Book 'The Hobbit' is already issued to someone else.

-- Returning a book --
  Book 'Dune' returned by Priya Mehta.

-- Available Fantasy books --
  [B005] 'The Hobbit Companion' by J.R.R. Tolkien (Fantasy) - Available

-- Members who have borrowed books --
  [M001] Aarav Shah - Borrowed: B001, B